# Match de personas: ConvAI2 × PersonaChat (`_revised`)

Este notebook compara personas completas de todos os splits de `visual-memory/ConvAI2` com as colunas revisadas de `visual-memory/PersonaChat`. A comparação é somente leitura e testa, em etapas, se o match é exato ou exige normalização.

> No PersonaChat, as listas estão serializadas como JSON. Fazer `json.loads` apenas adapta o formato e não altera o texto.

In [1]:
from __future__ import annotations

import json
import unicodedata
from collections.abc import Callable, Iterable
from typing import Any

import pandas as pd
from datasets import DatasetDict, load_dataset

CONVAI2_REPO = "visual-memory/ConvAI2"
PERSONACHAT_REPO = "visual-memory/PersonaChat"
CONVAI2_COLUMNS = ("your_persona", "partner_persona")
PERSONACHAT_COLUMNS = ("your_persona_revised", "partner_persona_revised")

convai2 = load_dataset(CONVAI2_REPO)
personachat = load_dataset(PERSONACHAT_REPO)

display(convai2)
display(personachat)

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'your_persona', 'partner_persona', 'turns', 'source_split'],
        num_rows: 17878
    })
    validation: Dataset({
        features: ['conversation_id', 'your_persona', 'partner_persona', 'turns', 'source_split'],
        num_rows: 1000
    })
})

DatasetDict({
    train: Dataset({
        features: ['dialog_id', 'your_persona_original', 'partner_persona_original', 'your_persona_revised', 'partner_persona_revised', 'dialog', 'your_enhanced_persona_original', 'partner_enhanced_persona_original', 'your_enhanced_persona_revised', 'partner_enhanced_persona_revised'],
        num_rows: 8939
    })
    validation: Dataset({
        features: ['dialog_id', 'your_persona_original', 'partner_persona_original', 'your_persona_revised', 'partner_persona_revised', 'dialog', 'your_enhanced_persona_original', 'partner_enhanced_persona_original', 'your_enhanced_persona_revised', 'partner_enhanced_persona_revised'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['dialog_id', 'your_persona_original', 'partner_persona_original', 'your_persona_revised', 'partner_persona_revised', 'dialog', 'your_enhanced_persona_original', 'partner_enhanced_persona_original', 'your_enhanced_persona_revised', 'partner_enhanced_persona_revised']

## Validação e adaptação de formato

In [2]:
def validate_columns(dataset: DatasetDict, required: Iterable[str]) -> None:
    required = set(required)
    missing = {
        split: sorted(required - set(data.column_names))
        for split, data in dataset.items()
        if required - set(data.column_names)
    }
    assert not missing, f"Colunas ausentes: {missing}"


def parse_persona(value: Any) -> tuple[str, ...]:
    """Adapta list[str] ou uma lista JSON para uma tupla, sem mudar o texto."""
    if isinstance(value, str):
        value = json.loads(value)
    assert isinstance(value, (list, tuple)), f"Persona inválida: {type(value)!r}"
    assert all(isinstance(sentence, str) for sentence in value)
    assert value, "Persona vazia"
    assert all(sentence.strip() for sentence in value), "Frase de persona vazia"
    return tuple(value)


def collect_values(dataset: DatasetDict, columns: Iterable[str]) -> list[tuple[str, ...]]:
    return [
        parse_persona(value)
        for data in dataset.values()
        for column in columns
        for value in data[column]
    ]


validate_columns(convai2, CONVAI2_COLUMNS)
validate_columns(personachat, PERSONACHAT_COLUMNS)

convai2_values = collect_values(convai2, CONVAI2_COLUMNS)
personachat_values = collect_values(personachat, PERSONACHAT_COLUMNS)

print(f"Ocorrências ConvAI2: {len(convai2_values):,}")
print(f"Ocorrências PersonaChat revised: {len(personachat_values):,}")

Ocorrências ConvAI2: 37,756
Ocorrências PersonaChat revised: 21,814


## Comparação progressiva

Cada etapa acumula somente uma mudança: igualdade exata; `strip`; independência da ordem das frases; e, por fim, normalização textual relaxada.

In [3]:
PersonaKey = tuple[str, ...]


def exact(persona: PersonaKey) -> PersonaKey:
    return persona


def stripped(persona: PersonaKey) -> PersonaKey:
    return tuple(sentence.strip() for sentence in persona)


def unordered(persona: PersonaKey) -> PersonaKey:
    return tuple(sorted(stripped(persona)))


def relaxed(persona: PersonaKey) -> PersonaKey:
    normalized = (
        " ".join(unicodedata.normalize("NFKC", sentence).casefold().split())
        for sentence in persona
    )
    return tuple(sorted(normalized))


def compare(normalizer: Callable[[PersonaKey], PersonaKey], stage: str) -> dict[str, Any]:
    left = {normalizer(persona) for persona in convai2_values}
    right = {normalizer(persona) for persona in personachat_values}
    intersection = left & right
    return {
        "etapa": stage,
        "convai2_unicas": len(left),
        "personachat_unicas": len(right),
        "matches": len(intersection),
        "somente_convai2": len(left - right),
        "somente_personachat": len(right - left),
        "cobertura_convai2_pct": round(100 * len(intersection) / len(left), 2),
        "cobertura_personachat_pct": round(100 * len(intersection) / len(right), 2),
    }


stages = [
    ("1. exata (ordem preservada)", exact),
    ("2. strip (ordem preservada)", stripped),
    ("3. frases sem ordem", unordered),
    ("4. NFKC + casefold + espaços", relaxed),
]
results = pd.DataFrame(compare(function, label) for label, function in stages)
display(results)

,etapa,convai2_unicas,personachat_unicas,matches,somente_convai2,somente_personachat,cobertura_convai2_pct,cobertura_personachat_pct
0,1. exata (ordem preservada),19601,20932,8513,11088,12419,43.43,40.67
1,2. strip (ordem preservada),19601,20932,8513,11088,12419,43.43,40.67
2,3. frases sem ordem,10321,6162,3787,6534,2375,36.69,61.46
3,4. NFKC + casefold + espaços,10321,6162,3787,6534,2375,36.69,61.46


## Exemplos dos matches e das diferenças

A etapa canônica representa uma persona como a tupla ordenada de suas frases, sem alterar caixa ou pontuação.

In [4]:
convai2_canonical = {unordered(persona) for persona in convai2_values}
personachat_canonical = {unordered(persona) for persona in personachat_values}
matched = convai2_canonical & personachat_canonical
only_convai2 = convai2_canonical - personachat_canonical
only_personachat = personachat_canonical - convai2_canonical


def examples(values: set[PersonaKey], limit: int = 3) -> pd.DataFrame:
    sample = sorted(values)[:limit]
    return pd.DataFrame(
        {"persona": ["\n".join(f"- {sentence}" for sentence in persona) for persona in sample]}
    )


print("Matches:")
display(examples(matched))
print("Somente no ConvAI2:")
display(examples(only_convai2))
print("Somente no PersonaChat revised:")
display(examples(only_personachat))

Matches:


,persona
0,"- a few months ago , i purchased an rv.\n- i a..."
1,"- a few months ago , i purchased an rv.\n- i a..."
2,"- a few months ago , i purchased an rv.\n- i a..."


Somente no ConvAI2:


,persona
0,"- a few months ago , i purchased an rv.\n- i a..."
1,"- a few months ago , i purchased an rv.\n- i h..."
2,"- a few months ago , i purchased an rv.\n- i h..."


Somente no PersonaChat revised:


,persona
0,- a few people share my flat.\n- i am an anima...
1,- a few people share my flat.\n- i am an anima...
2,- a few people share my flat.\n- i am an anima...


## Validações e conclusão

In [5]:
exact_row, strip_row, unordered_row, relaxed_row = results.to_dict(orient="records")

assert exact_row["matches"] == strip_row["matches"], (
    "`strip` alterou os matches; revise a conclusão."
)
assert unordered_row["matches"] == relaxed_row["matches"], (
    "A normalização textual relaxada alterou os matches; revise a conclusão."
)
assert (
    len(convai2_canonical), len(personachat_canonical), len(matched),
    len(only_convai2), len(only_personachat)
) == (10321, 6162, 3787, 6534, 2375)

print("Conclusão:")
print("- O parse de JSON é necessário apenas para compatibilizar os formatos.")
print("- `strip` não acrescenta matches à igualdade exata ordenada.")
print("- Para identificar personas, é necessário ignorar a ordem das frases.")
print("- NFKC, casefold e normalização de espaços não acrescentam matches.")
print(f"- Match canônico: {len(matched):,} personas.")
print(f"- Somente ConvAI2: {len(only_convai2):,}; somente PersonaChat revised: {len(only_personachat):,}.")
print("- Logo, os conjuntos possuem sobreposição parcial, não correspondência total.")

Conclusão:
- O parse de JSON é necessário apenas para compatibilizar os formatos.
- `strip` não acrescenta matches à igualdade exata ordenada.
- Para identificar personas, é necessário ignorar a ordem das frases.
- NFKC, casefold e normalização de espaços não acrescentam matches.
- Match canônico: 3,787 personas.
- Somente ConvAI2: 6,534; somente PersonaChat revised: 2,375.
- Logo, os conjuntos possuem sobreposição parcial, não correspondência total.
